# Microsoft Agent Framework — Azure OpenAI (Responses API)

V tomto ukážkovom kóde použijete **Microsoft Agent Framework (MAF)** na vytvorenie jednoduchého agenta založeného na **Azure OpenAI** pomocou **Responses API**.

> **Poznámka k migrácii:** Tento príklad predtým používal Semantic Kernel s GitHub Modelmi. Bol migrovaný na Microsoft Agent Framework a GitHub Modely (zastaralé, ukončenie júl 2026) boli nahradené Azure OpenAI, ktorý podporuje Responses API. `OpenAIChatClient` v MAF cieli na stabilný koncový bod Azure OpenAI `/openai/v1/` a ako predvolený používa Responses API.

Cieľom tohto príkladu je demonštrovať kroky, ktoré budú neskôr použité v ďalších ukážkach pri implementácii rôznych agentných vzorov.


In [ ]:
%pip install agent-framework agent-framework-openai azure-identity -q


## Importujte potrebné Python balíčky


In [ ]:
import os
import random

from dotenv import load_dotenv
from IPython.display import display, HTML

from agent_framework import tool
from agent_framework.openai import OpenAIChatClient
from azure.identity import AzureCliCredential


## Definovanie nástroja

V rámci Microsoft Agent Framework je **nástroj** obyčajná Python funkcia ozdobená dekorátorom `@tool`, ktorú môže agent volať. Nižšie definujeme nástroj, ktorý vracia náhodnú dovolenkovú destináciu a vyhýba sa opakovaniu tej predchádzajúcej.


In [ ]:
# A list of vacation destinations the tool can choose from.
_DESTINATIONS = [
    "Barcelona, Spain",
    "Paris, France",
    "Berlin, Germany",
    "Tokyo, Japan",
    "Sydney, Australia",
    "New York, USA",
    "Cairo, Egypt",
    "Cape Town, South Africa",
    "Rio de Janeiro, Brazil",
    "Bali, Indonesia",
]

# Track the last destination so repeated calls avoid immediate repeats.
_last_destination: str | None = None


@tool(approval_mode="never_require")
def get_random_destination() -> str:
    """Provides a random vacation destination."""
    global _last_destination
    available = _DESTINATIONS.copy()
    if _last_destination and len(available) > 1:
        available.remove(_last_destination)
    destination = random.choice(available)
    _last_destination = destination
    return destination


In [ ]:
load_dotenv()

endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# OpenAIChatClient targets Azure OpenAI's v1 endpoint and uses the Responses API.
# Sign in with `az login` first so AzureCliCredential can authenticate.
chat_client = OpenAIChatClient(
    model=deployment,
    azure_endpoint=endpoint,
    credential=AzureCliCredential(),
)


## Vytváranie agenta

Tu vytvárame agenta s názvom `TravelAgent`.

V tomto príklade používame veľmi základné inštrukcie. Neváhajte tieto inštrukcie upraviť a sledovať, ako sa zmení správanie agenta.


In [ ]:
agent = chat_client.as_agent(
    name="TravelAgent",
    instructions="You are a helpful AI Agent that can help plan vacations for customers at random destinations",
    tools=[get_random_destination],
)


## Spúšťanie agenta

Teraz môžeme spustiť agenta. Vytvoríme `AgentSession`, aby si agent pamätal konverzáciu medzi jednotlivými kolami, potom pošleme dve `user_inputs`. Prvá žiada o výlet; druhá hovorí, že sa používateľovi návrh nepáčil a žiada o iný — agent používa históriu relácie plus nástroj `get_random_destination` na odpoveď.

Tieto správy môžete upraviť, aby ste videli, ako agent reaguje odlišne. Odpovede sa **prenášajú** token po tokene.


In [ ]:
user_inputs = [
    "Plan me a day trip.",
    "I don't like that destination. Plan me another vacation.",
]


async def main():
    # A session keeps conversation history across turns.
    session = agent.create_session()

    for user_input in user_inputs:
        html_output = (
            f"<div style='margin-bottom:10px'>"
            f"<div style='font-weight:bold'>User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )

        full_response: list[str] = []
        # Stream the agent's response token-by-token. The agent will call the
        # get_random_destination tool automatically when it needs a destination.
        async for chunk in agent.run(user_input, session=session, stream=True):
            full_response.append(str(chunk))

        html_output += (
            "<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold'>TravelAgent:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
        )

        display(HTML(html_output))


await main()


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vyhlásenie o zodpovednosti**:
Tento dokument bol preložený pomocou AI prekladateľskej služby [Co-op Translator](https://github.com/Azure/co-op-translator). Hoci sa snažíme o presnosť, vezmite prosím na vedomie, že automatické preklady môžu obsahovať chyby alebo nepresnosti. Pôvodný dokument v jeho natívnom jazyku by mal byť považovaný za autoritatívny zdroj. Pre kritické informácie sa odporúča profesionálny ľudský preklad. Nie sme zodpovední za žiadne nedorozumenia alebo nesprávne interpretácie vyplývajúce z použitia tohto prekladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
